# Caso E · 02 ETL ERA5 → CAPTIA weather_station

> _Tutorial · Caso de uso: **E — Meteo & solar** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Convertir el mock ERA5 a `captia_point` con `domain_id=weather_station`. Aplicar conversiones unitarias y validar rangos físicos.


## 2. Qué se aprende

- Conversión K→°C, J/m²→W/m², m→mm.
- Cómo asignar `asset_id=era5_gridpoint`.
- Validaciones físicas previas a la escritura.


## 3. Contexto del caso de uso

Los datos meteo se usan en B y H.


## 4. Relación con CENTINELA+

Site `xativa` independiente del aula.


## 5. Relación con Medallion

Bronce → plata.


## 6. Datos de entrada

`era5_xativa_mock.csv`.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

5 variables: `temperature_outdoor`, `solar_irradiance`, `wind_speed`, `precipitation`, `pressure`.


## 9. Carga de datos o mock

Cargamos.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "era5_xativa_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df.head()


,timestamp,t_air_c,ghi_w_m2,wind_speed_ms,precip_mm,pressure_hpa
0,2024-06-01 00:00:00+00:00,19.82,0.0,1.00,0.00,1016.7
1,2024-06-01 01:00:00+00:00,18.68,0.0,2.54,0.00,1015.6
2,2024-06-01 02:00:00+00:00,21.06,0.0,0.00,0.01,1015.6
3,2024-06-01 03:00:00+00:00,22.21,0.0,4.16,0.00,1014.5
4,2024-06-01 04:00:00+00:00,20.56,0.0,1.78,0.01,1018.8


## 10. Exploración paso a paso

Mapping unidades.


In [3]:
mapping = {
    "t_air_c": "temperature_outdoor",
    "ghi_w_m2": "solar_irradiance",
    "wind_speed_ms": "wind_speed",
    "precip_mm": "precipitation",
    "pressure_hpa": "pressure",
}
print(mapping)


{'t_air_c': 'temperature_outdoor', 'ghi_w_m2': 'solar_irradiance', 'wind_speed_ms': 'wind_speed', 'precip_mm': 'precipitation', 'pressure_hpa': 'pressure'}


## 11. Transformación bronce → plata

Generamos line protocol.


In [4]:
out_dir = ROOT / "output" / "case_E"
out_dir.mkdir(parents=True, exist_ok=True)
lines = []
for _, row in df.iterrows():
    ts_ns = int(pd.Timestamp(row["timestamp"]).value)
    for csv_col, captia_var in mapping.items():
        lines.append(build_line_protocol(
            measurement=MEASUREMENT_TELEMETRY,
            tags={
                "captia_env": "dev", "domain_id": "weather_station",
                "site_id": "xativa", "asset_id": "era5_gridpoint",
                "variable": captia_var,
            },
            fields={"value": float(row[csv_col])},
            timestamp_ns=ts_ns,
        ))
(out_dir / "era5_xativa.lp").write_text("\n".join(lines), encoding="utf-8")
print(f"{len(lines)} líneas escritas")


3600 líneas escritas


## 12. Construcción de capa oro

Notebook 04.


## 13. Visualizaciones explicativas

Resumen anual sintético.


In [5]:
df.set_index("timestamp")[["t_air_c", "ghi_w_m2"]].plot(figsize=(10, 3))
plt.title("ERA5 Xàtiva mock 30 días")
plt.tight_layout()


## 14. Validaciones

Los rangos físicos se respetan en `captia_point_meta`.


In [6]:
for csv_col, var in mapping.items():
    rmin, rmax = KNOWN_VARIABLES[var]["range"]
    s = df[csv_col]
    assert s.between(rmin - 5, rmax + 5).all(), f"{var} fuera de rango"
print("Rangos físicos OK")


Rangos físicos OK


## 15. Errores comunes

1. Confundir `solar_irradiance` (GHI) con `solar_irradiance_dni`.
2. No restar 273.15 si trabajas con K.
3. Confundir tasa (J/s/m²) con energía acumulada (J/m²).


## 16. Ejercicios propuestos

1. Añade `dewpoint` calculada con la fórmula de Magnus.
2. Mapea `total_cloud_cover` (NetCDF real).
3. Construye un downsample diario.


## 17. Cómo se reutiliza con datos reales

Cambiar el path al NetCDF real y `xarray.open_dataset` en vez de `pd.read_csv`.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `05_case_E_weather_solar/03_features_meteorologicas.ipynb`.
- Documento web del caso: `docs/contracts/variable-catalog.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo de irradiancia solar global (Iqbal 1983)

$$
G_h(t) = G_b(t) + G_d(t), \quad G_b(t) = G_{sc} \cdot \tau_b(t) \cdot \cos\theta_z(t)
$$

con $G_{sc} = 1361$ W/m² constante solar y $\theta_z$ ángulo cenital del sol:

$$
\cos\theta_z = \sin\delta \sin\phi + \cos\delta \cos\phi \cos\omega
$$

donde $\delta$ es declinación solar, $\phi$ latitud (Xátiva 38.99°N) y
$\omega$ ángulo horario.

### Clear-sky index

$$
k_c(t) = \frac{G_h(t)}{G_{clear}(t)} \in [0, 1]
$$

separa astronomía (determinista) de meteorología (estocástica).

### Predictor XGBoost para FV

$$
\hat{P}(t+h) = P_{nominal} \cdot \eta_{panel} \cdot \text{XGB}(k_c(t), T_{out}, t_{hora}, t_{año})
$$

### Métrica Skill Score

$$
\text{Skill} = 1 - \frac{\text{RMSE}_{model}}{\text{RMSE}_{persistence}}
$$

Objetivo Simarro: $\text{nMAE} \leq 8\%$ a 24 h, $\text{Skill} \geq 0.3$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Predicción solar permite optimizar despacho energético en centros con FV instalada y planificar climatización aprovechando picos de radiación. CAPTIA puede ofrecer este caso como **producto añadido** a centros con paneles.

### ROI estimado

| Concepto | Valor |
|---|---|
| Optimización despacho FV (centro 50 kWp) | +800 €/año |
| Sinergia con Caso B forecast | +500 €/año |
| Coste integración ERA5+AEMET | -1 200 € one-time |
| **Payback** | **~12 meses** |


## 21. Bibliografía y referencias

- Iqbal, M. (1983). *An Introduction to Solar Radiation*. Academic Press.
- ECMWF (2024). *ERA5 Reanalysis Documentation*. Copernicus Climate Change Service.
- AEMET. *Open Data Portal*. https://opendata.aemet.es
- Holmgren, W. F. et al. (2018). *pvlib python: a python package for modeling solar energy systems*. JOSS 3(29).
